In [1]:
import pandas as pd
from darts.timeseries import TimeSeries
from darts.utils.timeseries_generation import datetime_attribute_timeseries
from darts.dataprocessing.transformers import Scaler
from darts.models import TFTModel
from darts.dataprocessing.transformers import StaticCovariatesTransformer
import numpy as np
import torch
import matplotlib.pyplot as plt
import joblib
import os

c:\Users\G0004878\Desktop\Virtual_environments\darts_gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The StatsForecast module could not be imported. To enable support for the AutoARIMA, AutoETS and Croston models, please consider installing it.
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md


### Loss Logger 

In [2]:
from pytorch_lightning.callbacks import Callback

class LossLogger(Callback):
    """
    A PyTorch Lightning callback to record training and validation losses 
    at the end of every epoch for custom plotting or analysis.
    """
    def __init__(self):
        super().__init__()
        self.train_losses = []
        self.val_losses = []

    def on_train_epoch_end(self, trainer, pl_module):
        # Retrieve train_loss from callback_metrics
        train_loss = trainer.callback_metrics.get("train_loss")
        
        if train_loss is not None:
            # detach() ensures we don't keep the computation graph in memory
            # cpu() ensures it works regardless of whether you're on GPU or CPU
            self.train_losses.append(float(train_loss.detach().cpu()))

    def on_validation_epoch_end(self, trainer, pl_module):
        # Retrieve val_loss from callback_metrics
        val_loss = trainer.callback_metrics.get("val_loss")
        
        if val_loss is not None:
            self.val_losses.append(float(val_loss.detach().cpu()))

### Add encoders

In [3]:
# Function to encode the year as a normalized value
def encode_year(idx):
  return (idx.year - 2000) / 50

def encode_days_in_month(index):
  return index.days_in_month.to_numpy().reshape(-1,1)

# Set up the add_encoders dictionary to specify how different time-related encoders and transformers should be applied
add_encoders = {
    'cyclic': {'past': ['month'], 'future': ['month']},
    'position': {'past': ['relative'], 'future': ['relative']},
    'custom': {
        'past': [encode_year, encode_days_in_month],
        'future': [encode_year, encode_days_in_month]
    },
    'transformer': Scaler()
}

### Read the data

In [23]:
pandas_df = pd.read_csv(r"C:\Users\G0004878\Desktop\TFT_Data\Multi_series\12_month_forecast\Pipeline\Input_data_preparation\Preparing the input data\Filtered_data_for_training.csv",index_col = ['MONTH_OF_SALE'],parse_dates=True)

In [24]:
pandas_df.shape

(1720047, 61)

In [25]:
pandas_df.groupby("PARENT_DEALER_CODE_MODEL_FAMILY").agg(COUNT_OF_RECORDS=('NET_SALES','count')).sort_values(by='COUNT_OF_RECORDS',ascending=True)

,COUNT_OF_RECORDS
PARENT_DEALER_CODE_MODEL_FAMILY,
10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,49
11657<>PASSION<>DRUM<>SELF<>CAST<>SPORTS RED,49
11657<>PASSION<>DRUM<>SELF<>CAST<>RED,49
11657<>PASSION<>DRUM<>SELF<>CAST<>BLUE,49
11657<>PASSION<>DRUM<>SELF<>CAST<>BLACK,49
...,...
10836<>HF DELUXE<>DRUM<>SELF<>CAST<>BLUE,49
10836<>HF DELUXE<>DRUM<>SELF<>CAST<>RED BLACK,49
10836<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLACK,49


### Separating the type of variables

In [26]:
#Extracting type of columns according to the datatypes
# 1. Targets/Metrics (The numbers we want to predict)
target_cols = pandas_df.select_dtypes(include=['number']).columns.tolist()
target_cols.append('PARENT_DEALER_CODE_MODEL_FAMILY')

# 2. Time Dimension
time_cols = pandas_df.select_dtypes(include=['datetime', 'datetime64']).columns.tolist()

# 3. Static/Categorical Covariates (The identifiers)
# We exclude numbers and dates to find the "ID" strings
static_cols = pandas_df.select_dtypes(exclude=['number', 'datetime', 'datetime64']).columns.tolist()

print(f"Targets: {target_cols}")
print(f"Time Column: {time_cols}")
print(f"Static Identifiers: {static_cols}")

Targets: ['PARENT_DEALER_CODE', 'NET_SALES', 'DUSSEHRA_(VIJAYADASHAMI)_DAYS', 'NUM_FESTIVE_DAYS_MONTH', 'AKSHAYA_TRITIYA_DAYS', 'BHAI_DOOJ_DAYS', 'BUDDHA_PURNIMA_DAYS', 'CHHATH_PUJA_DAYS', 'DHANTERAS_DAYS', 'DIWALI_DAYS', 'EID_UL_FITR_DAYS', 'GANESH_CHATURTHI_DAYS', 'GANGA_DUSSEHRA_DAYS', 'GOVARDHAN_POOJA_DAYS', 'GURU_PURNIMA_DAYS', 'HANUMAN_JAYANTI_DAYS', 'HARTALIK_TEEJ_DAYS', 'HOLI_DAYS', 'HOLIKA_DAHAN_DAYS', 'JAGANNATH_RATHYATRA_DAYS', 'JANMASHTAMI_DAYS', 'KARWA_CHAUTH_DAYS', 'LOHRI_DAYS', 'MAHA_SHIVARATRI_DAYS', 'MAKAR_SANKRANTI_PONGAL_DAYS', 'NAG_PANCHAMI_DAYS', 'NAVRATRI_DAYS', 'NEW_YEAR_DAYS', 'ONAM_DAYS', 'PITRAPAKSHA_DAYS', 'RAKSHA_BANDHAN_DAYS', 'REPUBLIC_DAY_DAYS', 'VASANT_PANCHAMI_DAYS', 'VISHWAKARMA_PUJA_DAYS', 'MARRIAGE_DAYS', 'FESTIVE_PHASE_I', 'FESTIVE_PHASE_II', 'FESTIVE_PHASE_III', 'PITRU_PAKSH', 'TOTAL_DAYS_FESTIVE_PHASE_I', 'TOTAL_DAYS_FESTIVE_PHASE_II', 'TOTAL_DAYS_FESTIVE_PHASE_III', 'TOTAL_DAYS_PITRU_PAKSH', 'PROP_FESTIVE_PHASE_I', 'PROP_EVENT_FESTIVE_PHASE_I', '

In [27]:
#Separating the covariates
target_col = ["NET_SALES"]

#future covariates
future_covariates = [i for i in target_cols if i!='NET_SALES']

#actual_static_cols
actual_static_cols = [i for i in static_cols if i!='PARENT_DEALER_CODE_MODEL_FAMILY']

In [28]:
actual_static_cols

['MODEL_FAMILY',
 'BRAKE_TYPE',
 'IGNITION_TYPE',
 'WHEEL_TYPE',
 'COLOUR',
 'DEALER_CITY',
 'X_CITY_CATEGORY',
 'ZONAL_OFFICE_NAME']

In [29]:
# #since variables like MODEL_FAMILY,BRAKE_VARIANT,IGNITION_TYPE,WHEEL_TYPE,BIKE_COLOUR are mostly same for all the top 10 series, will be removing them from the static covariates'
# static_covariates = [i for i in actual_static_cols if i not in ['MODEL_FAMILY','BRAKE_VARIANT','IGNITION_TYPE','WHEEL_TYPE','BIKE_COLOUR','DEALER_CODE']]
# static_covariates

static_covariates = actual_static_cols.copy()

static_covariates

['MODEL_FAMILY',
 'BRAKE_TYPE',
 'IGNITION_TYPE',
 'WHEEL_TYPE',
 'COLOUR',
 'DEALER_CITY',
 'X_CITY_CATEGORY',
 'ZONAL_OFFICE_NAME']

### Preparing data for Darts

In [30]:
#Step 1 - Sorting the dataframe by date
pandas_df=pandas_df.reset_index().sort_values(by=["PARENT_DEALER_CODE_MODEL_FAMILY","MONTH_OF_SALE"]).set_index("MONTH_OF_SALE")

In [31]:
#Step 2 - Separating the static covariates and NET_SALES column
pandas_df_with_target_and_static_covariates = pandas_df.loc[:,['PARENT_DEALER_CODE_MODEL_FAMILY','NET_SALES']+static_covariates]
pandas_df_with_target_and_static_covariates.head()

,PARENT_DEALER_CODE_MODEL_FAMILY,NET_SALES,MODEL_FAMILY,BRAKE_TYPE,IGNITION_TYPE,WHEEL_TYPE,COLOUR,DEALER_CITY,X_CITY_CATEGORY,ZONAL_OFFICE_NAME
MONTH_OF_SALE,,,,,,,,,,
2023-04-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,4.0,DESTINI,DRUM,SELF,CAST,BLACK,JANDIALA GURU,RURAL,Zonal Office - North
2023-05-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,6.0,DESTINI,DRUM,SELF,CAST,BLACK,JANDIALA GURU,RURAL,Zonal Office - North
2023-06-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,4.0,DESTINI,DRUM,SELF,CAST,BLACK,JANDIALA GURU,RURAL,Zonal Office - North
2023-07-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,5.0,DESTINI,DRUM,SELF,CAST,BLACK,JANDIALA GURU,RURAL,Zonal Office - North
2023-08-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,3.0,DESTINI,DRUM,SELF,CAST,BLACK,JANDIALA GURU,RURAL,Zonal Office - North


In [32]:
#Step 3 - Separating the future covariates
pandas_df_with_future_covariates = pandas_df.loc[:,future_covariates]
pandas_df_with_future_covariates.head()

,PARENT_DEALER_CODE,DUSSEHRA_(VIJAYADASHAMI)_DAYS,NUM_FESTIVE_DAYS_MONTH,AKSHAYA_TRITIYA_DAYS,BHAI_DOOJ_DAYS,BUDDHA_PURNIMA_DAYS,CHHATH_PUJA_DAYS,DHANTERAS_DAYS,DIWALI_DAYS,EID_UL_FITR_DAYS,...,PROP_FESTIVE_PHASE_I,PROP_EVENT_FESTIVE_PHASE_I,PROP_FESTIVE_PHASE_II,PROP_EVENT_FESTIVE_PHASE_II,PROP_FESTIVE_PHASE_III,PROP_EVENT_FESTIVE_PHASE_III,PROP_PITRU_PAKSH,PROP_EVENT_PITRU_PAKSH,NON_ZERO_FLAG,PARENT_DEALER_CODE_MODEL_FAMILY
MONTH_OF_SALE,,,,,,,,,,,,,,,,,,,,,
2023-04-01,10001,0.0,0.0,1,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK
2023-05-01,10001,0.0,0.0,0,0,1,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK
2023-06-01,10001,0.0,0.0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK
2023-07-01,10001,0.0,0.0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK
2023-08-01,10001,0.0,0.0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK


In [33]:
#Step 4 - Creating the darts timeseries object for target and static covariates
darts_df_with_static_covariates = TimeSeries.from_group_dataframe(df=pandas_df_with_target_and_static_covariates,
                                                                  group_cols=["PARENT_DEALER_CODE_MODEL_FAMILY"],
                                                                  static_cols=static_covariates,value_cols=["NET_SALES"],freq='MS')


In [34]:
#Step 5 - Creating the darts timeseries object with future covariates

#Removing PARENT_DEALER_CODE_MODEL_FAMILY from future_covariates
try:
    future_covariates.remove('PARENT_DEALER_CODE_MODEL_FAMILY')
except:
    pass

darts_df_with_future_covariates = TimeSeries.from_group_dataframe(df = pandas_df_with_future_covariates,
                                    group_cols="PARENT_DEALER_CODE_MODEL_FAMILY",
                                    freq = 'MS',
                                    value_cols = future_covariates
                                    )

### Train/Test split

In [42]:
#Step 6 - Creating train, test, and validation split
#Train set - Apr'23 to Dec'25 
#Val set - Jan'26 to Mar'26 


train_list = []
val_list = []

for ts in darts_df_with_static_covariates:
    train = ts.slice(pd.Timestamp('2023-04-01'), pd.Timestamp('2025-12-01'))
    val = ts.slice(pd.Timestamp('2025-01-01'), pd.Timestamp('2026-03-01'))
    
    train_list.append(train)
    val_list.append(val)

train_future_covariates_list = []
validation_future_covariates_list = []

for ts in darts_df_with_future_covariates:
    train = ts.slice(pd.Timestamp('2023-04-01'), pd.Timestamp('2025-12-01'))
    val = ts.slice(pd.Timestamp('2025-01-01'), pd.Timestamp('2026-03-01'))
    train_future_covariates_list.append(train)
    validation_future_covariates_list.append(val)

### Scaling

In [43]:

target_scaler = Scaler(n_jobs=-1)
future_covariates_scaler = Scaler(n_jobs=-1)

transformer = StaticCovariatesTransformer(n_jobs=-1)

#Scale the target training data
scaled_target_series = target_scaler.fit_transform(train_list)

scaled_target_series_with_static_covariates_training = transformer.fit_transform(scaled_target_series)



# #Scale the static covariates in training data
# scaled_static_covariates_training = transformer.fit_transform(train_list)

# #Scale the future covariates in training data
# # scaled_future_covariates = future_covariates_scaler.fit_transform(darts_df_with_future_covariates)

scaled_future_covariates_training = future_covariates_scaler.fit_transform(train_future_covariates_list)
scaled_future_covariates_validation = future_covariates_scaler.transform(validation_future_covariates_list)


# #Scale the target validation data
scaled_target_series_validation = target_scaler.transform(val_list)
scaled_target_series_with_static_covariates_validation = transformer.transform(scaled_target_series_validation)

# #Scale the static covariates in validation data
# scaled_static_covariates_validation = transformer.transform(val_list)



In [44]:
from datetime import datetime

from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from darts.models import TFTModel

In [45]:
loss_logger=LossLogger()


# =========================
# 2. DEFINE PATHS & MODEL NAME
# =========================
now = datetime.now().strftime("%Y-%m-%d_%H_%M_%S")

WORK_DIR = r"C:\Users\G0004878\Desktop\TFT_Data\Multi_series\12_month_forecast\Pipeline\Model"
MODEL_NAME = f"tft_net_sales_{now}"

MODEL_DIR = os.path.join(WORK_DIR, MODEL_NAME)
CHECKPOINT_DIR = os.path.join(MODEL_DIR, "checkpoints")

# Create directories if not present
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("MODEL_DIR:", MODEL_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)


# =========================
# 3. CUSTOM CHECKPOINT CLASS
# =========================
class DateStampedCheckpoint(ModelCheckpoint):

    @property
    def state_key(self) -> str:
        return f"DateStampedCheckpoint_{self.monitor}_{self.dirpath}"


checkpoint_callback = DateStampedCheckpoint(
    dirpath=CHECKPOINT_DIR,
    filename=f"tft-best-{now}-{{epoch:02d}}-{{val_loss:.4f}}",
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    save_last=True,
    verbose=True
)


# =========================
# 4. EARLY STOPPING
# =========================
early_stop_callback = EarlyStopping(
    monitor="val_loss",
    patience=10,
    mode="min",
    verbose=True
)


# =========================
# 5. MODEL DEFINITION
# =========================
model = TFTModel(
    input_chunk_length=12,
    output_chunk_length=3,
    batch_size=512,
    dropout=0.1,
    likelihood=None,
    loss_fn=torch.nn.MSELoss(),
    n_epochs=100,
    random_state=42,
    add_encoders=add_encoders,

    model_name=MODEL_NAME,
    work_dir=WORK_DIR,

    # IMPORTANT: avoid conflict with Darts internal checkpointing
    save_checkpoints=False,
    force_reset=True,

    pl_trainer_kwargs={
        "callbacks": [
            loss_logger,
            checkpoint_callback,
            early_stop_callback
        ],
        "enable_checkpointing": True,
        "gradient_clip_val": 0.1
    }
)


# =========================
# 6. LR FINDER
# =========================
print("\nRunning LR Finder...")

lr_finder = model.lr_find(
    series=scaled_target_series_with_static_covariates_training,
    future_covariates=scaled_future_covariates_training,
)

suggested_lr = lr_finder.suggestion()
print("Suggested Learning Rate:", suggested_lr)

# Apply LR
model.lr = suggested_lr

MODEL_DIR: C:\Users\G0004878\Desktop\TFT_Data\Multi_series\12_month_forecast\Pipeline\Model\tft_net_sales_2026-05-18_02_18_29
CHECKPOINT_DIR: C:\Users\G0004878\Desktop\TFT_Data\Multi_series\12_month_forecast\Pipeline\Model\tft_net_sales_2026-05-18_02_18_29\checkpoints

Running LR Finder...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Finding best initial lr: 100%|██████████| 100/100 [00:57<00:00,  1.73it/s]
Restoring states from the checkpoint path at c:\Users\G0004878\Desktop\TFT_Data\Multi_series\12_month_forecast\Pipeline\Model\.lr_find_8e64ba8d-1e6d-4b66-804d-786527449d66.ckpt
Restored all states from the checkpoint at c:\Users\G0004878\Desktop\TFT_Data\Multi_series\12_month_forecast\Pipeline\Model\.lr_find_8e64ba8d-1e6d-4b66-804d-786527449d66.ckpt


Suggested Learning Rate: 0.0005248074602497723


In [55]:
from datetime import datetime
from pytorch_lightning.callbacks import ModelCheckpoint

current_date = datetime.now().strftime("%Y-%m-%d")

now = datetime.now().strftime("%Y-%m-%d_%H_%M_%S")

WORK_DIR = r"C:\Users\G0004878\Desktop\TFT_Data\Multi_series\12_month_forecast\Pipeline\Model"
MODEL_NAME = f"tft_net_sales_{now}"

In [52]:
# =========================
# 7. TRAINING WITH VALIDATION
# =========================
print("\nStarting Training...")

model.fit(
    series=scaled_target_series_with_static_covariates_training,
    future_covariates=scaled_future_covariates_training,
    val_series=scaled_target_series_with_static_covariates_validation,
    val_future_covariates=scaled_future_covariates_validation,
    verbose=True
)



Starting Training...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                              | Type                             | Params | Mode 
------------------------------------------------------------------------------------------------
0  | criterion                         | MSELoss                          | 0      | train
1  | train_criterion                   | MSELoss                          | 0      | train
2  | val_criterion                     | MSELoss                          | 0      | train
3  | train_metrics                     | MetricCollection                 | 0      | train
4  | val_metrics                       | MetricCollection                 | 0      | train
5  | input_embeddings                  | _MultiEmbedding                  | 0      | train
6  | static_covariates_vsn             | _VariableSelectionNetwork        | 5.7 K  | train
7  | encoder_vsn

Epoch 0: 100%|██████████| 1303/1303 [12:31<00:00,  1.73it/s, train_loss=0.0309, val_loss=0.0431]

Metric val_loss improved. New best score: 0.043
Epoch 0, global step 1303: 'val_loss' reached 0.04314 (best 0.04314), saving model to 'C:\\Users\\G0004878\\Desktop\\TFT_Data\\Multi_series\\12_month_forecast\\Pipeline\\Model\\tft_net_sales_2026-05-18_02_18_29\\checkpoints\\tft-best-2026-05-18_02_18_29-epoch=00-val_loss=0.0431.ckpt' as top 1


Epoch 1: 100%|██████████| 1303/1303 [12:41<00:00,  1.71it/s, train_loss=0.0259, val_loss=0.0423]

...

Epoch 11: 100%|██████████| 1303/1303 [13:56<00:00,  1.56it/s, train_loss=0.0236, val_loss=0.0407]

Epoch 11, global step 15636: 'val_loss' was not in top 1


Epoch 12:  14%|█▍        | 184/1303 [01:53<11:33,  1.61it/s, train_loss=0.0242, val_loss=0.0407] 


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

In [59]:
import torch
from darts.models import TFTModel

# 1. Get the path to your best custom checkpoint file
best_ckpt_path = checkpoint_callback.best_model_path
print("Loading directly from raw PyTorch Lightning checkpoint:", best_ckpt_path)

# 2. Re-instantiate a fresh template model with IDENTICAL architecture
# Crucial: Override n_epochs to 0 here so the upcoming fit step is instantaneous
loaded_model = TFTModel(
    input_chunk_length=12,
    output_chunk_length=3,
    batch_size=512,
    dropout=0.1,
    likelihood=None,
    loss_fn=torch.nn.MSELoss(),
    n_epochs=0, # Set to 0 to prevent training
    random_state=42,
    add_encoders=add_encoders,
    model_name=MODEL_NAME,
    work_dir=WORK_DIR,
    save_checkpoints=False,
    force_reset=True
)

# 3. Force public initialization via an instant 0-epoch fit
print("Initializing model structure natively...")
loaded_model.fit(
    series=scaled_target_series_with_static_covariates_training,
    future_covariates=scaled_future_covariates_training
)

# 4. Safely load the raw weights from the .ckpt file directly into the structure
checkpoint = torch.load(best_ckpt_path, map_location="cpu")
loaded_model.model.load_state_dict(checkpoint["state_dict"])

# 5. Restore the original parameter just in case you use it for further retraining
loaded_model.n_epochs = 100

print("Success! Model structure initialized and weights successfully injected.")

Loading directly from raw PyTorch Lightning checkpoint: C:\Users\G0004878\Desktop\TFT_Data\Multi_series\12_month_forecast\Pipeline\Model\tft_net_sales_2026-05-18_02_18_29\checkpoints\tft-best-2026-05-18_02_18_29-epoch=09-val_loss=0.0396.ckpt
Initializing model structure natively...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                              | Type                             | Params | Mode 
------------------------------------------------------------------------------------------------
0  | criterion                         | MSELoss                          | 0      | train
1  | train_criterion                   | MSELoss                          | 0      | train
2  | val_criterion                     | MSELoss                          | 0      | train
3  | train_metrics                     | MetricCollection                 | 0      | train
4  | val_metrics                       | MetricCollection                 | 0      | train
5  | input_embeddings                  | _MultiEmbedding                  | 0      | train
6  | static_covariates_vsn             | _VariableSelectionNetwork        | 5.7 K  | train
7  | encoder_vsn

Success! Model structure initialized and weights successfully injected.


In [60]:
# Generate the forecast
forecast_series = loaded_model.predict(
    n=3, # Predicting your 3-month output chunk
    series=scaled_target_series_with_static_covariates_training, # Gives the model the past 12 months of history
    future_covariates=scaled_future_covariates_validation # Gives the model the known future features (Festivals, Marriage days, etc.)
)

print("Forecast generated successfully!")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting DataLoader 0: 100%|██████████| 69/69 [00:24<00:00,  2.78it/s]
Forecast generated successfully!


In [61]:
# Inverse-transform predictions back to original scale
pred_series_inv = target_scaler.inverse_transform(forecast_series)
val_inv         = target_scaler.inverse_transform(scaled_target_series_with_static_covariates_validation)

In [62]:
# Build output DataFrame — one row per series per month
records = []

for actual, forecast, original_series in zip(val_inv, pred_series_inv, val_list):

    # Get the series label from unscaled val_list (retains original string value)
    series_name = original_series.static_covariates['PARENT_DEALER_CODE_MODEL_FAMILY'].values[0]

    months          = forecast.time_index
    actual_values   = actual.values().flatten()
    forecast_values = forecast.values().flatten()

    for month, act, pred in zip(months, actual_values, forecast_values):
        records.append({
            'MONTH_OF_SALE'                  : month,
            'PARENT_DEALER_CODE_MODEL_FAMILY' : series_name,
            'ACTUAL_SALES'                   : round(float(act),  2),
            'PREDICTED_SALES'                : round(float(pred), 2)
        })

df_output = pd.DataFrame(records)
df_output['MONTH_OF_SALE'] = pd.to_datetime(df_output['MONTH_OF_SALE']).dt.strftime('%Y-%m-%d')
df_output = df_output.sort_values(['PARENT_DEALER_CODE_MODEL_FAMILY', 'MONTH_OF_SALE']).reset_index(drop=True)

print(f'Output shape : {df_output.shape}')
print(f'Months       : {df_output["MONTH_OF_SALE"].unique()}')
print(f'Series count : {df_output["PARENT_DEALER_CODE_MODEL_FAMILY"].nunique()}')
df_output.head(10)

Output shape : (105309, 4)
Months       : ['2026-01-01' '2026-02-01' '2026-03-01']
Series count : 35103


,MONTH_OF_SALE,PARENT_DEALER_CODE_MODEL_FAMILY,ACTUAL_SALES,PREDICTED_SALES
0,2026-01-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,0.0,2.76
1,2026-02-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,0.0,3.03
2,2026-03-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,0.0,3.21
3,2026-01-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE,0.0,7.38
4,2026-02-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE,0.0,6.72
5,2026-03-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE,1.0,7.02
6,2026-01-01,10001<>DESTINI<>DRUM<>SELF<>SHEET METAL<>BLACK,2.0,9.38
7,2026-02-01,10001<>DESTINI<>DRUM<>SELF<>SHEET METAL<>BLACK,11.0,9.05
8,2026-03-01,10001<>DESTINI<>DRUM<>SELF<>SHEET METAL<>BLACK,10.0,10.16
9,2026-01-01,10001<>DESTINI<>DRUM<>SELF<>SHEET METAL<>RED,2.0,1.57


### Preparing the prediction data

In [64]:
pandas_df.index

DatetimeIndex(['2023-04-01', '2023-05-01', '2023-06-01', '2023-07-01',
               '2023-08-01', '2023-09-01', '2023-10-01', '2023-11-01',
               '2023-12-01', '2024-01-01',
               ...
               '2026-07-01', '2026-08-01', '2026-09-01', '2026-10-01',
               '2026-11-01', '2026-12-01', '2027-01-01', '2027-02-01',
               '2027-03-01', '2027-04-01'],
              dtype='datetime64[ns]', name='MONTH_OF_SALE', length=1720047, freq=None)

In [65]:
df_reset = pandas_df.reset_index()

prediction_df = df_reset[(df_reset['MONTH_OF_SALE'] >= '2026-04-01') & 
                         (df_reset['MONTH_OF_SALE'] <= '2026-06-01')]

prediction_df = prediction_df.set_index(pandas_df.index.names)

prediction_df.head()


,PARENT_DEALER_CODE,MODEL_FAMILY,PARENT_DEALER_CODE_MODEL_FAMILY,BRAKE_TYPE,IGNITION_TYPE,WHEEL_TYPE,COLOUR,NET_SALES,DUSSEHRA_(VIJAYADASHAMI)_DAYS,NUM_FESTIVE_DAYS_MONTH,...,PROP_FESTIVE_PHASE_II,PROP_EVENT_FESTIVE_PHASE_II,PROP_FESTIVE_PHASE_III,PROP_EVENT_FESTIVE_PHASE_III,PROP_PITRU_PAKSH,PROP_EVENT_PITRU_PAKSH,DEALER_CITY,X_CITY_CATEGORY,ZONAL_OFFICE_NAME,NON_ZERO_FLAG
MONTH_OF_SALE,,,,,,,,,,,,,,,,,,,,,
2026-04-01,10001,DESTINI,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,DRUM,SELF,CAST,BLACK,4.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,JANDIALA GURU,RURAL,Zonal Office - North,1
2026-05-01,10001,DESTINI,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,DRUM,SELF,CAST,BLACK,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,JANDIALA GURU,RURAL,Zonal Office - North,1
2026-06-01,10001,DESTINI,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,DRUM,SELF,CAST,BLACK,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,JANDIALA GURU,RURAL,Zonal Office - North,0
2026-04-01,10001,DESTINI,10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE,DRUM,SELF,CAST,WHITE,7.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,JANDIALA GURU,RURAL,Zonal Office - North,1
2026-05-01,10001,DESTINI,10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE,DRUM,SELF,CAST,WHITE,6.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,JANDIALA GURU,RURAL,Zonal Office - North,1


In [66]:
#Making sure prediction_df has the same series as that of training data
unique_series_in_training_data = pandas_df["PARENT_DEALER_CODE_MODEL_FAMILY"].unique().tolist()

unique_series_in_prediction_data = prediction_df["PARENT_DEALER_CODE_MODEL_FAMILY"].unique().tolist()

print(f"{len(set(unique_series_in_prediction_data) - set(unique_series_in_training_data))} series are available in prediction data but not in training data")

print(f"{len(set(unique_series_in_training_data) - set(unique_series_in_prediction_data))} series are available in training data but not in prediction data")

0 series are available in prediction data but not in training data
0 series are available in training data but not in prediction data


### Darts preparation for Prediction Data

In [18]:
prediction_df.index

DatetimeIndex(['2025-04-01', '2025-04-01', '2025-04-01', '2025-04-01',
               '2025-04-01', '2025-04-01', '2025-04-01', '2025-04-01',
               '2025-04-01', '2025-04-01',
               ...
               '2027-03-01', '2027-03-01', '2027-03-01', '2027-03-01',
               '2027-03-01', '2027-03-01', '2027-03-01', '2027-03-01',
               '2027-03-01', '2027-03-01'],
              dtype='datetime64[ns]', name='MONTH_OF_SALE', length=1602600, freq=None)

In [67]:
prediction_pandas_df = prediction_df.copy()

try:
    future_covariates.remove('PARENT_DEALER_CODE_MODEL_FAMILY')
except:
    pass


darts_prediction_series = TimeSeries.from_group_dataframe(
    df=prediction_df,
    group_cols="PARENT_DEALER_CODE_MODEL_FAMILY",
    static_cols=static_covariates, 
    value_cols=future_covariates,
    freq='MS'
)

### Scale the prediction data

In [68]:
scaled_temporal = future_covariates_scaler.transform(darts_prediction_series)

final_prediction_series = transformer.transform(scaled_temporal)

In [69]:
df_output.to_csv(r"Jan_26_to_Mar_26_prediction.csv")

In [71]:
final_prediction_series[0]

PARENT_DEALER_CODE  DUSSEHRA_(VIJAYADASHAMI)_DAYS  NUM_FESTIVE_DAYS_MONTH  AKSHAYA_TRITIYA_DAYS  BHAI_DOOJ_DAYS  ...  PROP_FESTIVE_PHASE_III  PROP_EVENT_FESTIVE_PHASE_III  PROP_PITRU_PAKSH  PROP_EVENT_PITRU_PAKSH  NON_ZERO_FLAG
MONTH_OF_SALE                                                                                                                   ...                                                                                                               
2026-04-01                    0.0                            0.0                     0.0                   1.0             0.0  ...                     0.0                           0.0               0.0                     0.0            1.0
2026-05-01                    0.0                            0.0                     0.0                   0.0             0.0  ...                     0.0                           0.0               0.0                     0.0            1.0
2026-06-01                    0.0                            0.0                     0.0                   0.0             0.0  ...                     0.0                           0.0               0.0                     0.0            0.0

shape: (3, 51, 1), freq: MS, size: 1.20 KB

Static covariates:
    static_covariates  PARENT_DEALER_CODE_MODEL_FAMILY  MODEL_FAMILY  BRAKE_TYPE  IGNITION_TYPE  WHEEL_TYPE  COLOUR  DEALER_CITY  X_CITY_CATEGORY  ZONAL_OFFICE_NAME
    global_components                              0.0           0.0         1.0            1.0         0.0     0.0        337.0              0.0                2.0

In [74]:
scaled_future_covariates_training[-1]

,PARENT_DEALER_CODE,DUSSEHRA_(VIJAYADASHAMI)_DAYS,NUM_FESTIVE_DAYS_MONTH,AKSHAYA_TRITIYA_DAYS,BHAI_DOOJ_DAYS,...,PROP_FESTIVE_PHASE_III,PROP_EVENT_FESTIVE_PHASE_III,PROP_PITRU_PAKSH,PROP_EVENT_PITRU_PAKSH,NON_ZERO_FLAG
MONTH_OF_SALE,,,,,,,,,,,
2023-04-01,0.0,0.0,0.000000,1.0,0.0,...,0.000000,0.0,0.0,0.0,0.0
2023-05-01,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0
2023-06-01,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0
2023-07-01,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0
2023-08-01,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
2025-08-01,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.0,0.0,0.0,1.0
2025-09-01,0.0,0.0,0.967742,0.0,0.0,...,0.000000,0.0,1.0,1.0,1.0
2025-10-01,0.0,1.0,1.000000,0.0,1.0,...,0.829512,1.0,0.0,0.0,1.0


In [76]:
scaled_future_covariates_validation[0]

,PARENT_DEALER_CODE,DUSSEHRA_(VIJAYADASHAMI)_DAYS,NUM_FESTIVE_DAYS_MONTH,AKSHAYA_TRITIYA_DAYS,BHAI_DOOJ_DAYS,...,PROP_FESTIVE_PHASE_III,PROP_EVENT_FESTIVE_PHASE_III,PROP_PITRU_PAKSH,PROP_EVENT_PITRU_PAKSH,NON_ZERO_FLAG
MONTH_OF_SALE,,,,,,,,,,,
2025-01-01,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0
2025-02-01,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0
2025-03-01,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0
2025-04-01,0.0,0.0,0.000000,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0
2025-05-01,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
2025-11-01,0.0,0.0,0.967742,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0
2025-12-01,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0
2026-01-01,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0


In [77]:
complete_future_covariates_list = [
    val_cov.concatenate(pred_cov, ignore_time_axis=False)
    for val_cov, pred_cov in zip(
        scaled_future_covariates_validation, 
        final_prediction_series
    )
]

In [79]:
final_prediction_series[0]

PARENT_DEALER_CODE  DUSSEHRA_(VIJAYADASHAMI)_DAYS  NUM_FESTIVE_DAYS_MONTH  AKSHAYA_TRITIYA_DAYS  BHAI_DOOJ_DAYS  ...  PROP_FESTIVE_PHASE_III  PROP_EVENT_FESTIVE_PHASE_III  PROP_PITRU_PAKSH  PROP_EVENT_PITRU_PAKSH  NON_ZERO_FLAG
MONTH_OF_SALE                                                                                                                   ...                                                                                                               
2026-04-01                    0.0                            0.0                     0.0                   1.0             0.0  ...                     0.0                           0.0               0.0                     0.0            1.0
2026-05-01                    0.0                            0.0                     0.0                   0.0             0.0  ...                     0.0                           0.0               0.0                     0.0            1.0
2026-06-01                    0.0                            0.0                     0.0                   0.0             0.0  ...                     0.0                           0.0               0.0                     0.0            0.0

shape: (3, 51, 1), freq: MS, size: 1.20 KB

Static covariates:
    static_covariates  PARENT_DEALER_CODE_MODEL_FAMILY  MODEL_FAMILY  BRAKE_TYPE  IGNITION_TYPE  WHEEL_TYPE  COLOUR  DEALER_CITY  X_CITY_CATEGORY  ZONAL_OFFICE_NAME
    global_components                              0.0           0.0         1.0            1.0         0.0     0.0        337.0              0.0                2.0

In [80]:
pred_series = loaded_model.predict(
    n=3,
    series=scaled_target_series_with_static_covariates_validation,
    future_covariates=complete_future_covariates_list
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting DataLoader 0: 100%|██████████| 69/69 [00:24<00:00,  2.78it/s]


In [81]:
actual_pred_series = target_scaler.inverse_transform(pred_series)

In [ ]:
pred_static_covariates = target_scaler.inverse_transform()

static_covariates,PARENT_DEALER_CODE_MODEL_FAMILY,MODEL_FAMILY,BRAKE_TYPE,IGNITION_TYPE,WHEEL_TYPE,COLOUR,DEALER_CITY,X_CITY_CATEGORY,ZONAL_OFFICE_NAME
NET_SALES,0.0,0.0,1.0,1.0,0.0,0.0,337.0,0.0,2.0


In [ ]:
# Build output DataFrame — one row per series per month
records = []

for actual, forecast, original_series in zip(val_inv, pred_series_inv, val_list):

    # Get the series label from unscaled val_list (retains original string value)
    series_name = original_series.static_covariates['PARENT_DEALER_CODE_MODEL_FAMILY'].values[0]

    months          = forecast.time_index
    actual_values   = actual.values().flatten()
    forecast_values = forecast.values().flatten()

    for month, act, pred in zip(months, actual_values, forecast_values):
        records.append({
            'MONTH_OF_SALE'                  : month,
            'PARENT_DEALER_CODE_MODEL_FAMILY' : series_name,
            'ACTUAL_SALES'                   : round(float(act),  2),
            'PREDICTED_SALES'                : round(float(pred), 2)
        })

df_output = pd.DataFrame(records)
df_output['MONTH_OF_SALE'] = pd.to_datetime(df_output['MONTH_OF_SALE']).dt.strftime('%Y-%m-%d')
df_output = df_output.sort_values(['PARENT_DEALER_CODE_MODEL_FAMILY', 'MONTH_OF_SALE']).reset_index(drop=True)

print(f'Output shape : {df_output.shape}')
print(f'Months       : {df_output["MONTH_OF_SALE"].unique()}')
print(f'Series count : {df_output["PARENT_DEALER_CODE_MODEL_FAMILY"].nunique()}')
df_output.head(10)

Mapping complete! Your verified data output:
  MONTH_OF_SALE  NET_SALES  PARENT_DEALER_CODE_MODEL_FAMILY  MODEL_FAMILY  \
0    2026-04-01   3.479740                              0.0           0.0   
1    2026-05-01   3.478307                              0.0           0.0   
2    2026-06-01   0.020259                              0.0           0.0   
3    2026-04-01   8.674775                              1.0           0.0   
4    2026-05-01   7.432969                              1.0           0.0   

   BRAKE_TYPE  IGNITION_TYPE  WHEEL_TYPE  COLOUR  DEALER_CITY  \
0         1.0            1.0         0.0     0.0        337.0   
1         1.0            1.0         0.0     0.0        337.0   
2         1.0            1.0         0.0     0.0        337.0   
3         1.0            1.0         0.0    25.0        337.0   
4         1.0            1.0         0.0    25.0        337.0   

   X_CITY_CATEGORY  ZONAL_OFFICE_NAME  
0              0.0                2.0  
1              0.0   